In [8]:

import pandas as pd

df = pd.read_csv('/content/sample_data/mnist_train_small.csv')


X = df.drop('6', axis=1)
y = df['6']

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


rf_model = RandomForestClassifier(random_state=42)


param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}


grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)


print(f"Best Parameters for Random Forest: {grid_search.best_params_}")


best_rf_model = grid_search.best_estimator_


Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters for Random Forest: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


In [14]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


logreg_model = LogisticRegression()
logreg_model.fit(X_train, y_train)
logreg_pred = logreg_model.predict(X_test)


xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)


rf_pred = best_rf_model.predict(X_test)


print("Logistic Regression Report:")
print(classification_report(y_test, logreg_pred))

print("XGBoost Report:")
print(classification_report(y_test, xgb_pred))

print("Random Forest Report:")
print(classification_report(y_test, rf_pred))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       575
           1       0.94      0.96      0.95       630
           2       0.83      0.85      0.84       588
           3       0.85      0.85      0.85       624
           4       0.86      0.88      0.87       590
           5       0.81      0.85      0.83       513
           6       0.93      0.92      0.92       603
           7       0.91      0.91      0.91       646
           8       0.85      0.79      0.82       581
           9       0.86      0.86      0.86       650

    accuracy                           0.88      6000
   macro avg       0.88      0.88      0.88      6000
weighted avg       0.88      0.88      0.88      6000

XGBoost Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       575
           1       0.98      0.99      0.98       630
           2       0.95      0.96 

In [15]:
print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, logreg_pred))

print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))


Logistic Regression Confusion Matrix:
[[531   0   8   7   3   4   8   3   7   4]
 [  0 604   3   3   1   7   1   0  10   1]
 [  6  10 497  14  10   5  11  12  15   8]
 [  5   2  16 530   3  30   3   7  23   5]
 [  1   3  11   5 522   5  13   3   1  26]
 [  4   4  11  24   8 434   5   2  14   7]
 [  5   1  21   2   6   9 557   1   1   0]
 [  1   5   5   7  12   1   1 586   3  25]
 [  9  13  27  20   4  31   3   2 459  13]
 [  4   3   1   9  36   7   0  25   8 557]]
XGBoost Confusion Matrix:
[[564   0   2   3   1   2   1   0   1   1]
 [  0 625   2   3   0   0   0   0   0   0]
 [  6   1 564   5   2   1   0   7   0   2]
 [  1   1   9 589   1   7   0   3   6   7]
 [  0   2   3   2 563   0   4   1   3  12]
 [  2   1   1   3   1 497   3   0   3   2]
 [  1   1   2   1   3   7 584   0   4   0]
 [  1   5   7   2   5   0   0 612   4  10]
 [  5   2   5   3   2   5   1   2 551   5]
 [  1   2   1   3  16   2   0  10   7 608]]
Random Forest Confusion Matrix:
[[563   0   2   1   0   0   5   0   4   0]

In [16]:
import pickle


with open('churn_prediction_model.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)


with open('churn_prediction_model.pkl', 'rb') as f:
    model = pickle.load(f)


loaded_model_pred = model.predict(X_test)
